In [0]:
# Gold Layer - Star Schema building with incremental load
# Reads from Silver Delta table and builds dimension and fact tables using Delta MERGE

In [0]:
# Configuration

storage_account_name = dbutils.secrets.get(scope="etl_sales_scope", key="STORAGE_ACCOUNT_NAME")
storage_account_key = dbutils.secrets.get(scope="etl_sales_scope", key="STORAGE_ACCOUNT_KEY")
container_name = dbutils.secrets.get(scope="etl_sales_scope", key="AZURE_CONTAINER_NAME")

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net"
silver_path = f"{base_path}/silver/sales"
gold_path = f"{base_path}/gold"

print("Configuration loaded successfully")

In [0]:
# Read Silver layer

df_silver = spark.read.format("delta").load(silver_path)
print(f"Silver table loaded: {df_silver.count()} rows")


In [0]:
# Build dimension tables

from pyspark.sql.functions import monotonically_increasing_id, current_timestamp

# dim_product
def build_dim_product(df):
    return df.select('product_category') \
             .distinct() \
             .withColumn('product_id', monotonically_increasing_id() + 1) \
             .withColumn('created_at', current_timestamp()) \
             .withColumn('updated_at', current_timestamp())

# dim_region
def build_dim_region(df):
    return df.select('customer_region') \
             .distinct() \
             .withColumn('region_id', monotonically_increasing_id() + 1) \
             .withColumn('created_at', current_timestamp()) \
             .withColumn('updated_at', current_timestamp())

# dim_payment
def build_dim_payment(df):
    return df.select('payment_method') \
             .distinct() \
             .withColumn('payment_method_id', monotonically_increasing_id() + 1) \
             .withColumn('created_at', current_timestamp()) \
             .withColumn('updated_at', current_timestamp())

# dim_date
def build_dim_date(df):
    return df.select(
        'order_date', 'day_num', 'month_num',
        'quarter_num', 'year_num', 'day_of_week'
    ).distinct() \
     .withColumn('date_id', monotonically_increasing_id() + 1)

dim_product = build_dim_product(df_silver)
dim_region = build_dim_region(df_silver)
dim_payment = build_dim_payment(df_silver)
dim_date = build_dim_date(df_silver)

print("Dimension tables built successfully")

In [0]:
# Build fact table

def build_fact_sales(df, dim_product, dim_region, dim_payment, dim_date):
    fact = df.join(
        dim_product.select('product_id', 'product_category'),
        on='product_category',
        how='left'
    ).join(
        dim_region.select('region_id', 'customer_region'),
        on='customer_region',
        how='left'
    ).join(
        dim_payment.select('payment_method_id', 'payment_method'),
        on='payment_method',
        how='left'
    ).join(
        dim_date.select('date_id', 'order_date'),
        on='order_date',
        how='left'
    )

    return fact.select(
        'order_id', 'product_id', 'region_id',
        'payment_method_id', 'date_id', 'price',
        'discount_percent', 'discounted_price',
        'quantity_sold', 'total_revenue', 'rating', 'review_count'
    ).withColumn('created_at', current_timestamp()) \
     .withColumn('updated_at', current_timestamp())

fact_sales = build_fact_sales(df_silver, dim_product, dim_region, dim_payment, dim_date)

print(f"Fact table built: {fact_sales.count()} rows")

In [0]:
# Incremental load with Delta MERGE

from delta.tables import DeltaTable
import os

def merge_dimension(df, path, merge_condition, update_columns):
    """
    Incremental load for dimension tables using Delta MERGE
    """
    if DeltaTable.isDeltaTable(spark, path):
        delta_table = DeltaTable.forPath(spark, path)
        
        delta_table.alias("target").merge(
            df.alias("source"),
            merge_condition
        ).whenMatchedUpdate(
            set={col: f"source.{col}" for col in update_columns}
        ).whenNotMatchedInsertAll() \
         .execute()
        
        print(f"Merged into existing Delta table: {path}")
    else:
        df.write \
          .format("delta") \
          .mode("overwrite") \
          .save(path)
        print(f"Created new Delta table: {path}")

# Merge dimensions
merge_dimension(
    df=dim_product,
    path=f"{gold_path}/dim_product",
    merge_condition="target.product_category = source.product_category",
    update_columns=["updated_at"]
)

merge_dimension(
    df=dim_region,
    path=f"{gold_path}/dim_region",
    merge_condition="target.customer_region = source.customer_region",
    update_columns=["updated_at"]
)

merge_dimension(
    df=dim_payment,
    path=f"{gold_path}/dim_payment",
    merge_condition="target.payment_method = source.payment_method",
    update_columns=["updated_at"]
)

merge_dimension(
    df=dim_date,
    path=f"{gold_path}/dim_date",
    merge_condition="target.order_date = source.order_date",
    update_columns=[]
)

print("All dimension tables merged successfully")


In [0]:
# Merge fact table

def merge_fact(df, path):
    """
    Incremental load for fact table using Delta MERGE
    """
    if DeltaTable.isDeltaTable(spark, path):
        delta_table = DeltaTable.forPath(spark, path)

        delta_table.alias("target").merge(
            df.alias("source"),
            "target.order_id = source.order_id"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()

        print(f"Merged into existing fact table: {path}")
    else:
        df.write \
          .format("delta") \
          .mode("overwrite") \
          .save(path)
        print(f"Created new fact table: {path}")

merge_fact(fact_sales, f"{gold_path}/fact_sales")

print("Fact table merged successfully")
